# T02 — Document Ingestion and Chunking

How raw documents become the chunks that T01's pipeline retrieves:

1. PDF parsing: MarkItDown vs docling
2. Chunking strategies: header-based, fixed-size, paragraph-aware
3. Chunk sizes and the embedding token limit
4. Tables and spreadsheets
5. Where a fact ends up under each strategy

No API key needed.

## Setup

In [ ]:
import re
import time

from IPython.display import display

from conversational_toolkit.chunking.excel_chunker import ExcelChunker
from conversational_toolkit.chunking.pdf_chunker import (
    MarkdownConverterEngine,
    PDFChunker,
)
from ki_kompass_rag.baseline_rag import DATA_DIR
from ki_kompass_rag.ingestion import (
    analyze_chunks,
    char_histogram,
    estimate_tokens,
    fixed_size_chunks,
    header_based_chunks,
    paragraph_aware_chunks,
)

SAMPLE_PDF = str(DATA_DIR / "EPD_pallet_relicyc_logypal1.pdf")

## 1. PDF Parsing: MarkItDown vs Docling

A PDF must be converted to text (here Markdown) before chunking. The parser decides which headings are detected, whether tables survive, and what text is lost.

| Engine | Approach | Strengths | Weaknesses |
|---|---|---|---|
| `markitdown` | Extracts the text layer | Fast, many formats (DOCX, PPTX) | Often loses headings and table structure |
| `docling` | Layout and table models | Headings, tables, reading order | Slower; downloads models on first use |

The pipeline uses docling with OCR off (all PDFs here have a text layer). `load_chunks()` also skips files over 20 MB, which can exhaust memory.

In [ ]:
chunker = PDFChunker(do_ocr=False)
markdown = {}
for engine in MarkdownConverterEngine:
    start = time.perf_counter()
    markdown[engine.value] = chunker.to_markdown(SAMPLE_PDF, engine=engine)
    seconds = time.perf_counter() - start
    print(
        f"{engine.value:<12}{len(markdown[engine.value]):>8,} chars  {seconds:5.1f} s"
    )

The docling time includes loading its models once. Compare headings and tables:

In [ ]:
def headings(md: str) -> list[str]:
    return [line.strip() for line in md.splitlines() if re.match(r"^#{1,4} ", line)]


def count_tables(md: str) -> int:
    lines = [line.lstrip() for line in md.splitlines()]
    return sum(
        line.startswith("|") and not prev.startswith("|")
        for prev, line in zip(["", *lines], lines)
    )


for name, md in markdown.items():
    print(f"{name}: {len(headings(md))} headings, {count_tables(md)} tables")
    for heading in headings(md)[:10]:
        print(f"  {heading}")

In [ ]:
table = []
for line in markdown["docling"].splitlines():
    if line.lstrip().startswith("|"):
        table.append(line)
    elif table:
        break
print("\n".join(table) or "No table found.")

## 2. Chunking Strategies

| Strategy | How it splits | Best for | Risk |
|---|---|---|---|
| Header-based | One chunk per Markdown heading | Structured documents (EPDs, policies) | Very large or tiny chunks if headings are sparse |
| Fixed-size | Every N characters, with overlap | Predictable sizes | Cuts mid-sentence or mid-table |
| Paragraph-aware | Merges paragraphs up to a target size | Flowing text | Sizes vary more than fixed-size |

In [ ]:
chunkings = {
    "header_based": header_based_chunks(SAMPLE_PDF),
    "fixed_size_800": fixed_size_chunks(SAMPLE_PDF, chunk_size=800, overlap=100),
    "paragraph_500": paragraph_aware_chunks(SAMPLE_PDF, target_chars=500),
}
for name, chunks in chunkings.items():
    print(analyze_chunks(chunks, name))

In [ ]:
for name, chunks in chunkings.items():
    display(char_histogram(chunks, title=name))

## 3. The Embedding Token Limit

An embedding model only reads a limited number of tokens; the rest of a longer chunk is ignored. `text-embedding-3-small` reads 8191 tokens, so the `>8191 tokens` count above is normally 0. Small local models are far stricter: `all-MiniLM-L6-v2` stops at 256 tokens (~1000 characters):

In [ ]:
for name, chunks in chunkings.items():
    over = sum(estimate_tokens(c.content) > 256 for c in chunks)
    print(f"{name:<15}{over:>4} / {len(chunks)} chunks over 256 tokens")

With such a model, use a smaller chunk size or fixed-size chunking to cap chunk length.

## 4. Tables and Spreadsheets

Docling renders PDF tables as Markdown tables, which stay intact inside header-based chunks:

In [ ]:
table_chunks = [c for c in chunkings["header_based"] if c.content.count("|") >= 4]
print(f"{len(table_chunks)} header-based chunks contain a table\n")
if table_chunks:
    print(table_chunks[0].content)

Excel files are chunked separately, one Markdown table per sheet:

In [ ]:
for c in ExcelChunker().make_chunks(str(DATA_DIR / "ART_product_overview.xlsx")):
    print(f"Sheet: {c.title}")
    print(c.content)

## 5. Where the Facts Land

A retriever can only return what a chunk contains. Which chunks mention the GWP, and how much context do they carry?

In [ ]:
terms = ["gwp", "global warming", "carbon"]
for name, chunks in chunkings.items():
    hits = [c for c in chunks if any(t in c.content.lower() for t in terms)]
    print(f"\n{name}: {len(hits)} / {len(chunks)} chunks mention {terms}")
    if hits:
        print(f"First hit ({len(hits[0].content)} chars): {hits[0].content[:500]!r}")

Header-based chunks keep a section together; fixed-size chunks may split a table from its heading.

## Summary

| | Header-based | Fixed-size | Paragraph-aware |
|---|---|---|---|
| Boundaries | Headings | Every N chars | Paragraph breaks |
| Size predictability | Low | High | Medium |
| Context per chunk | High | Medium (overlap helps) | Medium |

Docling with header-based chunking suits the structured EPDs and policies in this corpus. For less structured documents or small embedding models, fixed-size chunking is safer.